In [ ]:
from datetime import datetime, date
import polars as pl
from data_engineering.bts.read_bts_ontime_to_flights import get_bts_flights_for_day
from data_engineering.flights.sfdps_to_flights import get_sfdps_flights_day
target_date = date(2026,3,1)
df_bts_flights = get_bts_flights_for_day(target_date)
df_sfdps_flights = get_sfdps_flights_day(target_date)

Loading airports from /Volumes/T2-SSD/planequery/data/raw/ourairports/airports.csv
6 overlapping flights found for 2026-03-01


In [ ]:
import polars as pl

shared_icaos = set(df_bts_flights["icao"].unique()) & (set(df_sfdps_flights["icao"].unique()))
df_bts_flights = df_bts_flights.filter(pl.col("icao").is_in(shared_icaos))
df_sfdps_flights = df_sfdps_flights.filter(pl.col("icao").is_in(shared_icaos))
start_dt = datetime(target_date.year, target_date.month, target_date.day)

In [21]:
from flights.evaluation.american_evals import continental_usa_to_continental_usa
df_bts_flights = df_bts_flights.filter(pl.col("landing_time").dt.date() == target_date)
df_sfdps_flights = df_sfdps_flights.filter(pl.col("landing_time").dt.date() == target_date)
df_bts_flights = continental_usa_to_continental_usa(df_bts_flights)
df_sfdps_flights = continental_usa_to_continental_usa(df_sfdps_flights)

In [25]:
from flights.flights_comparison import df_flights_comparision
from datetime import timedelta
start_dt = datetime(target_date.year, target_date.month, target_date.day)
end_dt = start_dt + timedelta(days=1)
df = df_flights_comparision(df_bts_flights, df_sfdps_flights, start_dt, end_dt)

In [26]:
from flights.flights_comparison import df_flights_comparison_stats
df_flights_comparison_stats(df)

{'true_positive': 13463,
 'false_positive': 1330,
 'false_negative': 10520,
 'precision': 0.9100926113702427,
 'recall': 0.561355960472001,
 'f1': 0.6943985970703528,
 'takeoff_airport_ident_match_pct': 1.0,
 'landing_airport_ident_match_pct': 1.0,
 'airport_ident_match_pct': 1.0}

In [27]:
COLUMNS = ["icao", "takeoff_time", "landing_time", "takeoff_airport_ident", "landing_airport_ident", "takeoff_time_df1", "landing_time_df1", "takeoff_airport_ident_df1", "landing_airport_ident_df1", "match_status"] 
df = df.select(COLUMNS)

In [31]:
df = df.filter(
    (pl.col("match_status") == "both").all().over("icao")
)

In [37]:
from data_engineering.adsb.read_adsb import read_adsb
from flights.flights_match_adsb import get_matching_icaos_in_flights
df_adsb = read_adsb(target_date, icaos = df["icao"].unique().to_list())
df_gold = get_matching_icaos_in_flights(df, df_adsb, start_dt)